In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split

# Data Preparation
data_dir = "./COVID-19_Radiography_Dataset"
BATCH_SIZE = 32

train_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((128, 128)),
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

test_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

train_dataset_full = datasets.ImageFolder(root=data_dir, transform=train_transforms)
test_dataset_full = datasets.ImageFolder(root=data_dir, transform=test_transforms)

class_mapping = train_dataset_full.class_to_idx
print(f"Class Encoding: {class_mapping}")

targets = train_dataset_full.targets
indices = list(range(len(train_dataset_full)))

train_indices, temp_indices, train_targets, temp_targets = train_test_split(
    indices, targets, test_size=0.40, stratify=targets, random_state=42
)

val_indices, test_indices, _, _ = train_test_split(
    temp_indices, temp_targets, test_size=0.50, stratify=temp_targets, random_state=42
)

train_dataset = Subset(train_dataset_full, train_indices)
val_dataset = Subset(test_dataset_full, val_indices)
test_dataset = Subset(test_dataset_full, test_indices)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Data Splitted -> Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
print("DataLoaders are ready!")


# FFNN Model (Lightweight & Faster)
class FFNN_COVID(nn.Module):
    def __init__(self, num_classes=4): 
        super(FFNN_COVID, self).__init__()
        
        # Convolutional Feature Extractor (in_channels=1)
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.relu = nn.ReLU(inplace=True)
        self.dropout2d = nn.Dropout2d(0.25)
        
        # Flatten
        self.flatten = nn.Flatten()
        
# The layer sizes were reduced here to speed up training
        self.fc1 = nn.Linear(8192, 512)
        self.fc2 = nn.Linear(512, 128)
        self.fc3 = nn.Linear(128, num_classes) 
        
        self.bn1 = nn.BatchNorm1d(512)
        self.bn2 = nn.BatchNorm1d(128)
        self.dropout = nn.Dropout(0.5)
        
    def forward(self, x):
        # x: [Batch, 1, 128, 128]
        
        x = self.relu(self.conv1(x))
        x = self.pool(x)                    
        x = self.dropout2d(x)
        
        x = self.relu(self.conv2(x))
        x = self.pool(x)                    
        x = self.dropout2d(x)
        
        x = self.relu(self.conv3(x))
        
        x = F.adaptive_avg_pool2d(x, (8, 8))   
        
        # (Flatten = 8192)
        x = self.flatten(x)
        
        # FFNN
        x = self.relu(self.bn1(self.fc1(x)))
        x = self.dropout(x)
        
        x = self.relu(self.bn2(self.fc2(x)))
        x = self.dropout(x)
        
        x = self.fc3(x)
        
        return x

# Model
num_classes = len(class_mapping)
model = FFNN_COVID(num_classes=num_classes)

print(f"\nModel Created Successfully!")
print(f"Number of Classes: {num_classes}")

# test output
dummy_input = torch.randn(2, 1, 128, 128)
output = model(dummy_input)
print(f"Output Shape: {output.shape}")

Class Encoding: {'COVID': 0, 'Lung_Opacity': 1, 'Normal': 2, 'Viral Pneumonia': 3}
Data Splitted -> Train: 25398 | Val: 8466 | Test: 8466
DataLoaders are ready!

Model Created Successfully!
Number of Classes: 4
Output Shape: torch.Size([2, 4])


In [4]:
import torch.optim as optim

# 1. Define device (GPU if available, otherwise CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# 2. Define Loss function and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 5 # Number of training epochs

print(f"Starting training on device: {device}")

# 3. Training Loop
for epoch in range(EPOCHS):
    model.train() 
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    train_acc = 100 * correct / total
    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {running_loss/len(train_loader):.4f}, Accuracy: {train_acc:.2f}%")

# 4. Save the trained model weights
torch.save(model.state_dict(), 'ffnn_model.pth')
print("FFNN model saved successfully as 'ffnn_model.pth'!")

Starting training on device: cpu
Epoch [1/5], Loss: 1.0060, Accuracy: 56.63%
Epoch [2/5], Loss: 0.8948, Accuracy: 62.33%
Epoch [3/5], Loss: 0.8375, Accuracy: 64.86%
Epoch [4/5], Loss: 0.8065, Accuracy: 66.52%
Epoch [5/5], Loss: 0.7763, Accuracy: 68.40%
FFNN model saved successfully as 'ffnn_model.pth'!
